# Phase 2b: Entity分布分析 (v2)

**目标**：分析entities在16个标准relations中的分布，为Entity聚类提供数据支持

**输入**：
- Phase 1原始数据 (自动检测)
- Relation映射文件 (自动检测最新版本)

**输出**：
- 每个relation的entity统计
- 频率分布可视化
- 聚类策略建议

**更新**：
- v2: 去除hardcode，自动检测文件，支持16个relations

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path
import sys

# 设置绘图样式
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

## 0. 配置与文件检测

In [ ]:
# 项目根目录
PROJECT_ROOT = Path('../').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

print(f"项目根目录: {PROJECT_ROOT}")
print(f"Results目录: {RESULTS_DIR}")
print(f"Outputs目录: {OUTPUTS_DIR}")

# 检查目录是否存在
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 自动检测Phase 1数据文件
def find_phase1_data():
    """自动查找Phase 1提取结果文件"""
    possible_paths = [
        RESULTS_DIR / 'phase1_5percent_exploration.json',
        OUTPUTS_DIR / 'extraction' / 'phase1_results.json',
        OUTPUTS_DIR / 'phase1_results.json',
        PROJECT_ROOT / 'phase1_results.json',
    ]
    
    for path in possible_paths:
        if path.exists():
            return path
    
    # 如果没找到，搜索所有可能的文件
    for pattern in ['*phase1*.json', '*extraction*.json', '*pilot*.json']:
        matches = list(RESULTS_DIR.glob(pattern)) + list(OUTPUTS_DIR.rglob(pattern))
        if matches:
            return matches[0]
    
    return None

phase1_file = find_phase1_data()

if phase1_file:
    print(f"✓ 找到Phase 1数据: {phase1_file.relative_to(PROJECT_ROOT)}")
else:
    print("❌ 未找到Phase 1数据文件！")
    print("\n请确保以下位置之一存在Phase 1提取结果：")
    print("  - results/phase1_5percent_exploration.json")
    print("  - outputs/extraction/phase1_results.json")
    print("\n或手动指定：")
    print("  phase1_file = Path('your/path/to/phase1_data.json')")
    sys.exit(1)

In [ ]:
# 自动检测Relation映射文件（优先使用最新版本）
def find_relation_mapping():
    """自动查找Relation映射文件，优先v2版本"""
    possible_paths = [
        RESULTS_DIR / 'relation_mapping_final_v2.json',  # 最新LLM微调版本
        RESULTS_DIR / 'relation_mapping_final_v1.json',
        RESULTS_DIR / 'relation_mapping_final.json',
    ]
    
    for path in possible_paths:
        if path.exists():
            return path
    
    # 搜索所有可能的文件
    matches = list(RESULTS_DIR.glob('*relation_mapping*.json'))
    if matches:
        # 按修改时间排序，返回最新的
        return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    
    return None

relation_mapping_file = find_relation_mapping()

if relation_mapping_file:
    print(f"✓ 找到Relation映射: {relation_mapping_file.relative_to(PROJECT_ROOT)}")
else:
    print("❌ 未找到Relation映射文件！")
    print("\n请确保results/目录下存在relation_mapping文件")
    sys.exit(1)

## 1. 加载数据

In [ ]:
# 加载Phase 1原始数据
print(f"正在加载: {phase1_file}")
with open(phase1_file, 'r') as f:
    phase1_data_full = json.load(f)

# 提取results列表（支持不同的数据格式）
if 'results' in phase1_data_full:
    phase1_results = phase1_data_full['results']
elif isinstance(phase1_data_full, list):
    phase1_results = phase1_data_full
else:
    raise ValueError("无法识别的Phase 1数据格式")

print(f"\n✓ Phase 1数据: {len(phase1_results)} 部电影")

# 显示数据格式
if phase1_results:
    sample = phase1_results[0]
    print(f"\n数据格式示例:")
    for key in sample.keys():
        print(f"  {key}: {type(sample[key]).__name__}")

In [ ]:
# 加载Relation映射
print(f"正在加载: {relation_mapping_file}")
with open(relation_mapping_file, 'r') as f:
    relation_mapping_data = json.load(f)

relation_mapping = relation_mapping_data['relation_mapping']
standard_relations = relation_mapping_data['standard_relations']

print(f"\n✓ {len(standard_relations)}个标准Relations:")
for i, rel in enumerate(sorted(standard_relations), 1):
    print(f"{i:2d}. {rel}")

# 显示mapping统计
print(f"\n映射统计:")
print(f"  原始relations: {len(relation_mapping)}")
print(f"  标准relations: {len(standard_relations)}")

## 2. 提取所有知识点并映射到标准relations

In [ ]:
# 提取所有知识点（只处理成功提取的电影）
all_knowledge_points = []

for movie_result in phase1_results:
    # 兼容不同的status字段名
    status = movie_result.get('status', movie_result.get('extraction_status', 'success'))
    if status != 'success':
        continue
    
    # 兼容不同的ID字段名
    movie_id = movie_result.get('recbole_id', movie_result.get('movie_id', movie_result.get('id')))
    
    for kp in movie_result.get('knowledge_points', []):
        # 映射到标准relation
        original_relation = kp.get('relation')
        if not original_relation:
            continue
            
        # 使用最接近的标准relation（如果映射中不存在）
        standard_relation = relation_mapping.get(
            original_relation,
            relation_mapping.get(original_relation.lower(), 'additional_elements')
        )
        
        all_knowledge_points.append({
            'movie_id': movie_id,
            'original_relation': original_relation,
            'standard_relation': standard_relation,
            'entity': kp.get('entity')
        })

print(f"总知识点数: {len(all_knowledge_points)}")
print(f"\n前5个知识点示例:")
for i, kp in enumerate(all_knowledge_points[:5], 1):
    print(f"{i}. {kp['standard_relation']:20s} -> {kp['entity']}")

## 3. 统计每个标准relation的entity分布

In [ ]:
# 按标准relation分组统计entities
relation_entities = defaultdict(list)

for kp in all_knowledge_points:
    relation_entities[kp['standard_relation']].append(kp['entity'])

# 统计每个relation的详细信息
entity_stats = {}

for relation, entities in relation_entities.items():
    entity_counter = Counter(entities)
    unique_entities = len(entity_counter)
    total_instances = len(entities)
    
    # 频率分布统计
    freq_1 = sum(1 for count in entity_counter.values() if count == 1)
    freq_2_4 = sum(1 for count in entity_counter.values() if 2 <= count <= 4)
    freq_5_10 = sum(1 for count in entity_counter.values() if 5 <= count <= 10)
    freq_10_plus = sum(1 for count in entity_counter.values() if count > 10)
    
    entity_stats[relation] = {
        'unique_entities': unique_entities,
        'total_instances': total_instances,
        'entity_counter': entity_counter,
        'freq_distribution': {
            'freq_1': freq_1,
            'freq_2_4': freq_2_4,
            'freq_5_10': freq_5_10,
            'freq_10+': freq_10_plus
        },
        'top_entities': entity_counter.most_common(10)
    }

print("每个Relation的Entity统计:")
print("="*100)

# 按unique_entities排序显示
sorted_relations = sorted(entity_stats.items(), key=lambda x: x[1]['unique_entities'], reverse=True)

for relation, stats in sorted_relations:
    print(f"\n{relation}:")
    print(f"  Unique entities: {stats['unique_entities']:3d}")
    print(f"  Total instances: {stats['total_instances']:3d}")
    print(f"  频率分布: freq=1: {stats['freq_distribution']['freq_1']:3d}, "
          f"freq=2-4: {stats['freq_distribution']['freq_2_4']:3d}, "
          f"freq=5-10: {stats['freq_distribution']['freq_5_10']:3d}, "
          f"freq>10: {stats['freq_distribution']['freq_10+']:3d}")
    print(f"  Top 5 entities:")
    for entity, count in stats['top_entities'][:5]:
        print(f"    - {entity:40s}: {count:3d}")

## 4. 全局统计

In [ ]:
# 全局统计
total_unique_entities = sum(stats['unique_entities'] for stats in entity_stats.values())
total_instances = sum(stats['total_instances'] for stats in entity_stats.values())
num_relations = len(standard_relations)

# 统计所有entity的频率（跨relation统计全局唯一entity数）
all_entities = [kp['entity'] for kp in all_knowledge_points]
all_entity_counter = Counter(all_entities)
global_unique = len(all_entity_counter)

global_freq_1 = sum(1 for count in all_entity_counter.values() if count == 1)
global_freq_2_4 = sum(1 for count in all_entity_counter.values() if 2 <= count <= 4)
global_freq_5_plus = sum(1 for count in all_entity_counter.values() if count >= 5)

print("="*100)
print("全局统计")
print("="*100)
print(f"{num_relations}个标准relations")
print(f"总unique entities (按relation累加): {total_unique_entities}")
print(f"全局unique entities (跨relation去重): {global_unique}")
print(f"总instances: {total_instances}")
print(f"平均每个relation: {total_unique_entities / num_relations:.1f} unique entities")
print(f"\n全局频率分布 (跨relation统计):")
print(f"  只出现1次的entities: {global_freq_1} ({global_freq_1/global_unique*100:.1f}%)")
print(f"  出现2-4次的entities: {global_freq_2_4} ({global_freq_2_4/global_unique*100:.1f}%)")
print(f"  出现5次以上的entities: {global_freq_5_plus} ({global_freq_5_plus/global_unique*100:.1f}%)")
print(f"\nEntity复用率 (全局): {total_instances / global_unique:.2f}x")

## 5. 可视化

In [ ]:
# 图表A: 每个relation的entity数量（横向条形图）
fig, ax = plt.subplots(figsize=(14, max(10, len(sorted_relations) * 0.5)))

relations_list = [r for r, _ in sorted_relations]
unique_counts = [entity_stats[r]['unique_entities'] for r in relations_list]
total_counts = [entity_stats[r]['total_instances'] for r in relations_list]

y_pos = np.arange(len(relations_list))

# 绘制两个柱状图
bars1 = ax.barh(y_pos + 0.2, unique_counts, 0.4, label='Unique entities', color='steelblue')
bars2 = ax.barh(y_pos - 0.2, total_counts, 0.4, label='Total instances', color='coral', alpha=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels(relations_list)
ax.invert_yaxis()
ax.set_xlabel('Count')
ax.set_title(f'Entity数量分布（{len(relations_list)}个Relations）', fontsize=16, fontweight='bold')
ax.legend()

# 在柱子上标注数字
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    ax.text(bar1.get_width() + 2, bar1.get_y() + bar1.get_height()/2, 
            f'{int(bar1.get_width())}', va='center', fontsize=9)
    ax.text(bar2.get_width() + 2, bar2.get_y() + bar2.get_height()/2, 
            f'{int(bar2.get_width())}', va='center', fontsize=9)

plt.tight_layout()
output_path = RESULTS_DIR / 'entity_distribution_by_relation.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"图表已保存: {output_path.relative_to(PROJECT_ROOT)}")

In [ ]:
# 图表B: 频率分布堆叠柱状图
fig, ax = plt.subplots(figsize=(max(16, len(relations_list) * 0.8), 8))

relations_list = [r for r, _ in sorted_relations]
freq_1_list = [entity_stats[r]['freq_distribution']['freq_1'] for r in relations_list]
freq_2_4_list = [entity_stats[r]['freq_distribution']['freq_2_4'] for r in relations_list]
freq_5_10_list = [entity_stats[r]['freq_distribution']['freq_5_10'] for r in relations_list]
freq_10_plus_list = [entity_stats[r]['freq_distribution']['freq_10+'] for r in relations_list]

x = np.arange(len(relations_list))
width = 0.6

# 堆叠柱状图
p1 = ax.bar(x, freq_1_list, width, label='freq=1 (singleton)', color='#d62728')
p2 = ax.bar(x, freq_2_4_list, width, bottom=freq_1_list, label='freq=2-4', color='#ff7f0e')
p3 = ax.bar(x, freq_5_10_list, width, 
           bottom=np.array(freq_1_list) + np.array(freq_2_4_list), 
           label='freq=5-10', color='#2ca02c')
p4 = ax.bar(x, freq_10_plus_list, width, 
           bottom=np.array(freq_1_list) + np.array(freq_2_4_list) + np.array(freq_5_10_list), 
           label='freq>10', color='#1f77b4')

ax.set_xlabel('Relation')
ax.set_ylabel('Entity数量')
ax.set_title('Entity频率分布（堆叠图）', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(relations_list, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
output_path = RESULTS_DIR / 'entity_frequency_distribution.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"图表已保存: {output_path.relative_to(PROJECT_ROOT)}")

## 6. 聚类难度评估与策略建议

In [ ]:
# 根据entity数量分类
category_large = []   # > 60 entities
category_medium = []  # 20-60 entities
category_small = []   # < 20 entities

for relation, stats in entity_stats.items():
    unique = stats['unique_entities']
    if unique > 60:
        category_large.append((relation, unique, stats['total_instances']))
    elif unique >= 20:
        category_medium.append((relation, unique, stats['total_instances']))
    else:
        category_small.append((relation, unique, stats['total_instances']))

print("="*100)
print("聚类难度评估与策略建议")
print("="*100)

print(f"\n【需要重点聚类】(unique entities > 60):")
for rel, unique, total in sorted(category_large, key=lambda x: x[1], reverse=True):
    print(f"  - {rel:25s}: {unique:3d} entities ({total:3d} instances)")
    print(f"    建议: 聚类到 25-35 个标准entities")

print(f"\n【适度聚类】(20 ≤ unique entities ≤ 60):")
for rel, unique, total in sorted(category_medium, key=lambda x: x[1], reverse=True):
    print(f"  - {rel:25s}: {unique:3d} entities ({total:3d} instances)")
    target = int(unique * 0.5)
    print(f"    建议: 聚类到 {target} 个标准entities (压缩50%)")

print(f"\n【轻量处理】(unique entities < 20):")
for rel, unique, total in sorted(category_small, key=lambda x: x[1], reverse=True):
    freq_1 = entity_stats[rel]['freq_distribution']['freq_1']
    print(f"  - {rel:25s}: {unique:3d} entities ({total:3d} instances, {freq_1} singletons)")
    if freq_1 > unique * 0.5:
        print(f"    建议: 保留高频entities，合并singletons到others_entity")
    else:
        print(f"    建议: 直接保留所有，或轻量合并")

## 7. 计算聚类目标数量

In [ ]:
# 自适应聚类策略
clustering_targets = {}

for relation, stats in entity_stats.items():
    unique = stats['unique_entities']
    freq_1 = stats['freq_distribution']['freq_1']
    
    # 策略规则
    if unique < 10:
        # 极少entities，保留全部或只合并freq=1
        target = unique - freq_1 + 1  # 所有freq>1的 + 1个others
        strategy = 'preserve_high_freq'
    elif 10 <= unique < 30:
        # 小规模，轻量聚类
        target = max(int(unique * 0.6), 8)  # 压缩到60%，最少8个
        strategy = 'light_clustering'
    elif 30 <= unique < 80:
        # 中等规模，适度聚类
        target = 20
        strategy = 'moderate_clustering'
    else:  # >= 80
        # 大规模，重度聚类
        target = 30
        strategy = 'heavy_clustering'
    
    clustering_targets[relation] = {
        'current': unique,
        'target': target,
        'strategy': strategy,
        'compression_rate': target / unique if unique > 0 else 1.0
    }

print("="*100)
print("聚类目标规划")
print("="*100)
print(f"\n{'Relation':25s} {'当前':>6s} {'目标':>6s} {'压缩率':>8s} {'策略':20s}")
print("-"*100)

total_current = 0
total_target = 0

for relation in sorted(clustering_targets.keys()):
    info = clustering_targets[relation]
    total_current += info['current']
    total_target += info['target']
    
    print(f"{relation:25s} {info['current']:6d} {info['target']:6d} "
          f"{info['compression_rate']:7.1%} {info['strategy']:20s}")

print("-"*100)
print(f"{'总计':25s} {total_current:6d} {total_target:6d} {total_target/total_current:7.1%}")
print(f"\n全局压缩率: {total_current} → {total_target} ({total_target/total_current*100:.1f}%)")

## 8. 保存统计结果

In [ ]:
from datetime import datetime

# 保存统计数据供后续使用
output_data = {
    'metadata': {
        'total_knowledge_points': len(all_knowledge_points),
        'total_unique_entities_by_relation': total_unique_entities,
        'global_unique_entities': global_unique,
        'num_standard_relations': len(standard_relations),
        'analysis_date': datetime.now().strftime('%Y-%m-%d'),
        'phase1_file': str(phase1_file.relative_to(PROJECT_ROOT)),
        'relation_mapping_file': str(relation_mapping_file.relative_to(PROJECT_ROOT))
    },
    'entity_stats_by_relation': {
        relation: {
            'unique_entities': stats['unique_entities'],
            'total_instances': stats['total_instances'],
            'freq_distribution': stats['freq_distribution'],
            'top_10_entities': dict(stats['top_entities'])
        }
        for relation, stats in entity_stats.items()
    },
    'clustering_targets': clustering_targets,
    'summary': {
        'current_total': total_current,
        'target_total': total_target,
        'compression_rate': total_target / total_current
    }
}

output_path = RESULTS_DIR / 'entity_distribution_analysis.json'
with open(output_path, 'w') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"统计数据已保存: {output_path.relative_to(PROJECT_ROOT)}")

## 10. 详细Entity列表（用于判断relation分类质量和聚类策略）

In [ ]:
# 显示每个relation下的所有entities（按频率排序）
print("="*120)
print("各标准Relation下的Entity详细分布")
print("="*120)

for relation in sorted(entity_stats.keys()):
    stats = entity_stats[relation]
    entity_counter = stats['entity_counter']
    
    print(f"\n{relation} ({stats['unique_entities']} unique entities, {stats['total_instances']} total instances)")
    print(f"  Singleton率: {stats['freq_distribution']['freq_1']}/{stats['unique_entities']} = "
          f"{stats['freq_distribution']['freq_1']/stats['unique_entities']*100:.1f}%")
    print(f"  Entities (按频率排序):")
    
    # 按频率排序显示所有entities
    sorted_entities = sorted(entity_counter.items(), key=lambda x: (-x[1], x[0]))
    
    for entity, count in sorted_entities:
        marker = "⭐" if count > 5 else ("★" if count > 2 else " ")
        print(f"    {marker} {entity:50s} : {count:3d}")
    
    # 检测相似entity（可能需要合并）
    # 简单策略：找到包含相同词干的entities
    entity_names = list(entity_counter.keys())
    similar_groups = []
    
    for i, e1 in enumerate(entity_names):
        for e2 in entity_names[i+1:]:
            # 检测相似性（简单的包含关系或编辑距离小）
            if e1 in e2 or e2 in e1:
                similar_groups.append((e1, e2, entity_counter[e1], entity_counter[e2]))
    
    if similar_groups:
        print(f"\n  ⚠️  检测到相似entities（可能需要合并）:")
        for e1, e2, c1, c2 in similar_groups[:10]:  # 只显示前10组
            print(f"    - '{e1}' ({c1}) <-> '{e2}' ({c2})")
    
    # 根据当前entity数量给出聚类建议
    target_info = clustering_targets[relation]
    print(f"\n  💡 建议: {target_info['strategy']} - "
          f"当前{target_info['current']}个 → 目标{target_info['target']}个 "
          f"(压缩至{target_info['compression_rate']*100:.1f}%)")
    
    print("-"*120)